# MangoLeaf AI: threshold fitting, notebook revision v3.1

Reuse the calibrated **Trial 3 / model-v3.0.0** weights. This notebook performs inference only: no training, no temperature refit, and no model-weight export.

1. Choose margin and entropy using mango **`val/`** plus **OOD tune** images.
2. Save the chosen settings before running evaluation predictions.
3. Evaluate those fixed settings using mango **`test/`** plus **OOD report** images.

Confidence stays **0.70**. The selection target is at least **95% validation mango retention** and the highest OOD tuning rejection on the stated grid. Retention means a prediction is accepted; it does not mean the diagnosis is correct.

The original v3 test and OOD report sets already influenced earlier tuning. The default report explicitly identifies them as **previously used evaluation data**, not an independent final test. Fresh evaluation folders are supported through the paths below.

`v3.1` names this notebook revision. Under the repository's release policy, a threshold-only deployment would normally be **model-v3.0.1**. The candidate config uses that label; no deployed files change when this notebook runs.

## Before running: Google Drive inputs

Upload or reuse this layout. Keep the six class subfolders and their images inside both `val/` and `test/`.

```text
MyDrive/
  mangoleaf_v3/resume/
    mangoleaf_v3_calibrated.keras
    resume_info.json
  dataset_split/
    val/
      anthracnose/
      bacterial_black_spot/
      bacterial_canker/
      healthy/
      mango_scab/
      sooty_mold/
    test/
      anthracnose/
      bacterial_black_spot/
      bacterial_canker/
      healthy/
      mango_scab/
      sooty_mold/
  mango_ood_test/
    ...OOD image files...
```

- Existing counts: **391 validation**, **393 test**, **168 OOD** images. Counts can change for explicitly reviewed exclusions or fresh evaluation data.
- `train/`, the raw unsplit dataset, `y_prob_cal.npy`, `y_true.npy`, and browser `public/model/` shards are not needed.
- Reuse the original calibrated `.keras` file from the training notebook's Drive `resume/` folder. Renaming a TensorFlow.js `.json` or `.bin` file will not create this file.
- If OOD `tune/` and `report/` folders already exist, use them. Otherwise a flat image folder is split deterministically in memory, with seed 42. Images are never moved or deleted.
- Review OOD labels first. The supplied pool contains `Mango-Scab-Disease-Management-1.jpg`, which may be a supported mango disease. A filename or visual suspicion is not a verified diagnosis. If review confirms it is in scope, list its relative path in `OOD_EXCLUDE`; the original file remains untouched.
- Keep related photos of the same specimen/session in the same OOD subset. Exact-file checks cannot establish this automatically.

Open this notebook in Colab, choose a Python runtime (GPU optional), edit the paths in Step 1, and run cells in order. Each run creates a new results directory. To repeat the whole experiment, rerun from Step 1. Do not overwrite completed v3 evidence notebooks.

## Step 1: paths and runtime

Mount Drive and load the standard Colab TensorFlow stack. Dependency versions and model identity are recorded with the results. If loading fails, use a runtime compatible with the original saved Keras model; do not retrain just to recreate the resume file.

In [4]:
from google.colab import drive

drive.mount('/content/drive')

import csv
import hashlib
import json
import math
import platform
import uuid
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image, __version__ as pillow_version

RESUME_DIR = Path('/content/drive/MyDrive/mangoleaf_v3/resume')
DATASET_DIR = Path('/content/drive/MyDrive/mangoleaf_v3/dataset_split/dataset_split')
VAL_DIR = DATASET_DIR / 'val'
TEST_DIR = DATASET_DIR / 'test'
OOD_DIR = Path('/content/drive/MyDrive/mango_ood_test')
OUTPUT_ROOT = Path('/content/drive/MyDrive/mangoleaf_v3_1_threshold_results')

# Relative to OOD_DIR; exclude only after reviewing the image's true scope.
OOD_EXCLUDE = []

# Only change this for evaluation images never used in development or selection.
EVALUATION_STATUS = 'previously_used'  # Or 'fresh_reserved'.
EVALUATION_NOTES = 'Original v3 test/report images influenced earlier threshold work.'

CLASS_NAMES = ['anthracnose', 'bacterial_black_spot', 'bacterial_canker',
               'healthy', 'mango_scab', 'sooty_mold']
SEED = 42
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
CONFIDENCE_THRESHOLD = 0.70
MIN_ID_RETENTION = 0.95
GRID = [i / 100 for i in range(101)]
BASELINE_THRESHOLDS = {'confidenceThreshold': 0.70,
                       'marginThreshold': 0.40, 'entropyThreshold': 0.40}
CANDIDATE_VERSION = 'model-v3.0.1'
MODEL_PATH = RESUME_DIR / 'mangoleaf_v3_calibrated.keras'
INFO_PATH = RESUME_DIR / 'resume_info.json'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

if EVALUATION_STATUS not in {'previously_used', 'fresh_reserved'}:
    raise ValueError('Choose previously_used or fresh_reserved for EVALUATION_STATUS.')
if not EVALUATION_NOTES.strip():
    raise ValueError('Describe the evaluation image provenance in EVALUATION_NOTES.')
if CONFIDENCE_THRESHOLD != 0.70:
    raise ValueError('The project specifies a fixed 0.70 confidence threshold.')

# Outputs must never share a tree with an input folder.
for input_dir in [RESUME_DIR, VAL_DIR, TEST_DIR, OOD_DIR]:
    a, b = OUTPUT_ROOT.resolve(), input_dir.resolve()
    if a == b or a in b.parents or b in a.parents:
        raise ValueError(f'Output/input directories overlap: {input_dir}')

RUN_DIR = OUTPUT_ROOT / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
                         + '-' + uuid.uuid4().hex[:8])
RUN_DIR.mkdir(parents=True, exist_ok=False)
tf.keras.utils.set_random_seed(SEED)
RUNTIME = {'python': platform.python_version(), 'tensorflow': tf.__version__,
           'keras': tf.keras.__version__, 'numpy': np.__version__, 'pillow': pillow_version}
print('Runtime:', RUNTIME)
print('Outputs:', RUN_DIR)
print('Evaluation status:', EVALUATION_STATUS, '-', EVALUATION_NOTES)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: {'python': '3.13.15', 'tensorflow': '2.20.0', 'keras': '3.13.2', 'numpy': '2.1.3', 'pillow': '11.3.0'}
Outputs: /content/drive/MyDrive/mangoleaf_v3_1_threshold_results/20260916T195200Z-d75750ef
Evaluation status: previously_used - Original v3 test/report images influenced earlier threshold work.


## Step 2: validate files and record the OOD split

This reads all inputs to check image integrity and byte-identical overlaps. It does not run predictions on the final evaluation sets. Cross-subset duplicates and mango/OOD overlap stop the run. Same-subset duplicates are reported. Perceptually similar photos and disease labels still need human review.

For a flat OOD folder, assignment matches the old sorted-file, NumPy seed-42 split. Exclusions happen **after assignment**, so removing a reviewed image does not reshuffle all the remaining images. Explicit `tune/` and `report/` folders take priority.

In [5]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def write_json_once(path, value):
    with Path(path).open('x', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, allow_nan=False)
        handle.write('\n')


def list_images(folder):
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f'Image folder missing: {folder}')
    paths = sorted((p for p in folder.rglob('*')
                    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS),
                   key=lambda p: p.relative_to(folder).as_posix())
    if not paths:
        raise ValueError(f'No supported images in {folder}')
    return paths


def mango_records(folder, subset):
    directories = {p.name for p in folder.iterdir() if p.is_dir() and not p.name.startswith('.')}
    if directories != set(CLASS_NAMES):
        raise ValueError(f'{folder} must contain exactly these class folders: {CLASS_NAMES}')
    if any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in folder.iterdir()):
        raise ValueError(f'Move no files automatically: images in {folder} need class subfolders.')
    rows = []
    for label in CLASS_NAMES:
        for path in list_images(folder / label):
            rows.append({'subset': subset, 'path': str(path.resolve()),
                         'relativePath': path.relative_to(folder).as_posix(), 'label': label})
    return rows


def audit_records(rows):
    by_hash = {}
    same_subset_duplicates = []
    for row in rows:
        path = Path(row['path'])
        try:
            with Image.open(path) as picture:
                picture.convert('RGB').load()
        except Exception as error:
            raise ValueError(f'Unreadable image: {path}') from error
        row['sha256'] = sha256_file(path)
        previous = by_hash.get(row['sha256'])
        if previous:
            if (previous['subset'], previous['label']) != (row['subset'], row['label']):
                raise ValueError('Identical image crosses subsets/labels: '
                                 f"{previous['path']} and {row['path']}")
            same_subset_duplicates.append([previous['path'], row['path']])
        else:
            by_hash[row['sha256']] = row
    return same_subset_duplicates


def verify_records_unchanged(rows):
    for row in rows:
        if sha256_file(row['path']) != row['sha256']:
            raise RuntimeError(f"Image changed during this run: {row['path']}")

In [6]:
for required in [MODEL_PATH, INFO_PATH]:
    if not required.is_file():
        raise FileNotFoundError(f'Reuse the original Trial 3 resume file: {required}')
resume_info = json.loads(INFO_PATH.read_text(encoding='utf-8'))
if str(resume_info.get('trial')) != '3' or resume_info.get('classes') != CLASS_NAMES:
    raise ValueError('Resume metadata is not Trial 3 with the expected class order.')
if resume_info.get('temperatureFoldedIntoWeights') is not True:
    raise ValueError('Use the calibrated model with temperature already folded into its weights.')
if not math.isclose(float(resume_info['confidenceThreshold']), 0.70, abs_tol=1e-9):
    raise ValueError('Resume confidence threshold differs from the project specification.')
TEMPERATURE = float(resume_info['temperature'])
if not math.isfinite(TEMPERATURE) or TEMPERATURE <= 0:
    raise ValueError('Invalid temperature in resume metadata.')
MODEL_SHA256 = sha256_file(MODEL_PATH)
INFO_SHA256 = sha256_file(INFO_PATH)
with zipfile.ZipFile(MODEL_PATH) as archive:
    saved_keras_metadata = json.loads(archive.read('metadata.json'))
print('Saved Keras version:', saved_keras_metadata.get('keras_version'))
print('Model SHA-256:', MODEL_SHA256)

val_records = mango_records(VAL_DIR, 'val')
test_records = mango_records(TEST_DIR, 'test')
tune_dir, report_dir = OOD_DIR / 'tune', OOD_DIR / 'report'
if tune_dir.is_dir() != report_dir.is_dir():
    raise ValueError('OOD layout needs both tune/ and report/, or neither.')
if tune_dir.is_dir():
    tune_paths, report_paths = list_images(tune_dir), list_images(report_dir)
    if set(list_images(OOD_DIR)) != set(tune_paths + report_paths):
        raise ValueError('OOD images exist outside tune/ and report/. Review the folder layout.')
    split_method = 'existing tune/report folders'
else:
    all_paths = list_images(OOD_DIR)
    permutation = np.random.RandomState(SEED).permutation(len(all_paths))
    half = len(all_paths) // 2
    tune_paths = [all_paths[i] for i in sorted(permutation[:half])]
    report_paths = [all_paths[i] for i in sorted(permutation[half:])]
    split_method = 'sorted relative paths, NumPy RandomState seed 42, 50/50 in memory'

available = {p.relative_to(OOD_DIR).as_posix() for p in tune_paths + report_paths}
if set(OOD_EXCLUDE) - available:
    raise ValueError(f'Excluded OOD paths not found: {sorted(set(OOD_EXCLUDE) - available)}')

def ood_records(paths, subset):
    return [{'subset': subset, 'path': str(p.resolve()),
             'relativePath': p.relative_to(OOD_DIR).as_posix(), 'label': 'OOD'}
            for p in paths if p.relative_to(OOD_DIR).as_posix() not in OOD_EXCLUDE]

tune_records = ood_records(tune_paths, 'ood_tune')
report_records = ood_records(report_paths, 'ood_report')
if not tune_records or not report_records:
    raise ValueError('Each OOD subset must contain images after reviewed exclusions.')
all_records = val_records + test_records + tune_records + report_records
duplicate_pairs = audit_records(all_records)
manifest = {'seed': SEED, 'oodSplitMethod': split_method, 'oodExcluded': OOD_EXCLUDE,
            'evaluationStatus': EVALUATION_STATUS, 'evaluationNotes': EVALUATION_NOTES,
            'sameSubsetExactDuplicatePairs': duplicate_pairs, 'images': all_records}
MANIFEST_PATH = RUN_DIR / 'input_manifest.json'
write_json_once(MANIFEST_PATH, manifest)
MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)
for name, rows in [('val', val_records), ('test', test_records),
                   ('ood_tune', tune_records), ('ood_report', report_records)]:
    print(name, len(rows), dict(Counter(row['label'] for row in rows)))
print('Same-subset exact duplicate pairs:', len(duplicate_pairs))
print('Manifest:', MANIFEST_PATH)
print('Review OOD membership and specimen/session overlap before interpreting results.')

Saved Keras version: 3.13.2
Model SHA-256: 71dc9e47181598ef694d430a89d4ecd89e081b8dcfbf27dc12e1f5b202cc9bd8
val 391 {'anthracnose': 73, 'bacterial_black_spot': 44, 'bacterial_canker': 74, 'healthy': 91, 'mango_scab': 34, 'sooty_mold': 75}
test 393 {'anthracnose': 73, 'bacterial_black_spot': 45, 'bacterial_canker': 74, 'healthy': 92, 'mango_scab': 34, 'sooty_mold': 75}
ood_tune 84 {'OOD': 84}
ood_report 84 {'OOD': 84}
Same-subset exact duplicate pairs: 0
Manifest: /content/drive/MyDrive/mangoleaf_v3_1_threshold_results/20260916T195200Z-d75750ef/input_manifest.json
Review OOD membership and specimen/session overlap before interpreting results.


## Step 3: load the calibrated model and define scoring

Images are decoded to RGB with Pillow, resized to 224 x 224 by TensorFlow bilinear interpolation, and passed as raw float32 pixels in 0-255. MobileNetV3 handles rescaling internally; temperature is already folded into the loaded weights. No augmentation is applied.

These are **Keras runtime measurements**. Saved layer precision policies are preserved and recorded. Browser canvas decoding/resizing and TensorFlow.js WebGL arithmetic can differ; this notebook does not claim browser parity or simulate it by casting model weights to float16.

In [7]:
model = tf.keras.models.load_model(MODEL_PATH, compile=False, safe_mode=True)
if tuple(model.input_shape[1:]) != (224, 224, 3) or tuple(model.output_shape[1:]) != (6,):
    raise ValueError(f'Unexpected model shapes: {model.input_shape}, {model.output_shape}')


def weight_digest():
    digest = hashlib.sha256()
    for variable in model.weights:
        array = variable.numpy()
        digest.update(str(array.shape).encode())
        digest.update(str(array.dtype).encode())
        digest.update(array.tobytes())
    return digest.hexdigest()


def layer_policies(layer):
    policies = [str(layer.dtype_policy)]
    for child in getattr(layer, 'layers', []):
        policies.extend(layer_policies(child))
    return policies


WEIGHTS_SHA256 = weight_digest()
LAYER_POLICIES = dict(Counter(layer_policies(model)))
print('Saved layer policies:', LAYER_POLICIES)


def predict_records(records):
    predictions = []
    for start in range(0, len(records), BATCH_SIZE):
        tensors = []
        for row in records[start:start + BATCH_SIZE]:
            if sha256_file(row['path']) != row['sha256']:
                raise RuntimeError(f"Image changed since audit: {row['path']}")
            with Image.open(row['path']) as picture:
                pixels = np.asarray(picture.convert('RGB'), dtype=np.float32)
            tensors.append(tf.image.resize(pixels, IMAGE_SIZE, method='bilinear',
                                           preserve_aspect_ratio=False, antialias=False))
        output = model(tf.stack(tensors), training=False).numpy()
        predictions.extend(output.tolist())
    if len(predictions) != len(records):
        raise RuntimeError('Prediction count does not match the image manifest.')
    return predictions

Saved layer policies: {'<DTypePolicy "float32">': 194}


In [8]:
def probability_stats(probabilities):
    stats = []
    for row in probabilities:
        values = [float(p) for p in row]
        if len(values) != len(CLASS_NAMES) or any(
                not math.isfinite(p) or p < 0 or p > 1 for p in values):
            raise ValueError('Expected six finite probabilities in [0, 1].')
        if not math.isclose(sum(values), 1.0, abs_tol=1e-4):
            raise ValueError('Expected softmax probabilities summing to one.')
        ordered = sorted(values, reverse=True)
        stats.append({'confidence': ordered[0], 'margin': ordered[0] - ordered[1],
                      'entropy': -sum(p * math.log(p) for p in values if p > 0)
                                 / math.log(len(values)),
                      'predictedIndex': values.index(ordered[0])})
    return stats


def accepted(stat, thresholds):
    return (stat['confidence'] >= thresholds['confidenceThreshold']
            and stat['margin'] >= thresholds['marginThreshold']
            and stat['entropy'] <= thresholds['entropyThreshold'])


def select_thresholds(val_stats, ood_tune_stats, grid, confidence, min_retention):
    # Only tuning data enters this function. Counts avoid rounded-score ties.
    if not val_stats or not ood_tune_stats:
        raise ValueError('Both tuning subsets must contain predictions.')
    if not 0 < min_retention <= 1 or not 0 <= confidence <= 1:
        raise ValueError('Invalid confidence or retention target.')
    if not grid or any(not math.isfinite(x) or not 0 <= x <= 1 for x in grid):
        raise ValueError('Grid must contain finite values between zero and one.')
    required = math.ceil(min_retention * len(val_stats) - 1e-9)
    best, best_key, rows = None, None, []
    for margin in grid:
        val_eligible = [s for s in val_stats if s['confidence'] >= confidence and s['margin'] >= margin]
        ood_eligible = [s for s in ood_tune_stats if s['confidence'] >= confidence and s['margin'] >= margin]
        for entropy in grid:
            kept = sum(s['entropy'] <= entropy for s in val_eligible)
            rejected = len(ood_tune_stats) - sum(s['entropy'] <= entropy for s in ood_eligible)
            feasible = kept >= required
            rows.append({'margin': margin, 'entropy': entropy, 'valAccepted': kept,
                         'valTotal': len(val_stats), 'oodRejected': rejected,
                         'oodTotal': len(ood_tune_stats), 'meetsRetentionTarget': feasible})
            # Ties: more mango retention, then lower margin, then higher entropy.
            key = (rejected, kept, -margin, entropy)
            if feasible and (best_key is None or key > best_key):
                best_key = key
                best = {'confidenceThreshold': confidence, 'marginThreshold': margin,
                        'entropyThreshold': entropy}
    if best is None:
        raise ValueError('No grid pair meets the validation retention target. '
                         'Review tuning data or the prespecified target; '
                         'no replacement thresholds were invented.')
    return best, rows


def scored_rows(records, stats, thresholds):
    if len(records) != len(stats):
        raise ValueError('Image and prediction counts differ.')
    rows = []
    for record, stat in zip(records, stats):
        keep = accepted(stat, thresholds)
        prediction = CLASS_NAMES[stat['predictedIndex']]
        reasons = []
        if stat['confidence'] < thresholds['confidenceThreshold']: reasons.append('confidence')
        if stat['margin'] < thresholds['marginThreshold']: reasons.append('margin')
        if stat['entropy'] > thresholds['entropyThreshold']: reasons.append('entropy')
        rows.append({**record, **stat, 'predictedClass': prediction, 'accepted': keep,
                     'returnedClass': prediction if keep else 'uncertain',
                     'rejectionReasons': ','.join(reasons),
                     'acceptedCorrect': keep and prediction == record['label']})
    return rows


def summarize(rows):
    if not rows:
        raise ValueError('Cannot report an empty subset.')
    count = len(rows)
    kept = sum(row['accepted'] for row in rows)
    report = {'total': count, 'accepted': kept, 'rejected': count - kept,
              'retention': kept / count, 'rejectionRate': (count - kept) / count}
    if all(row['label'] == 'OOD' for row in rows):
        report['falseAcceptRate'] = kept / count
        return report
    if any(row['label'] == 'OOD' for row in rows):
        raise ValueError('Report mango and OOD subsets separately.')
    correct = sum(row['acceptedCorrect'] for row in rows)
    report.update({'rawAccuracy': sum(row['predictedClass'] == row['label'] for row in rows) / count,
                   'acceptedCorrect': correct, 'acceptedWrong': kept - correct,
                   'acceptedAccuracy': correct / kept if kept else None,
                   'acceptedErrorsPerInput': (kept - correct) / count})
    per_class = {}
    confusion = [[0] * (len(CLASS_NAMES) + 1) for _ in CLASS_NAMES]
    for row in rows:
        column = row['predictedIndex'] if row['accepted'] else len(CLASS_NAMES)
        confusion[CLASS_NAMES.index(row['label'])][column] += 1
    for label in CLASS_NAMES:
        group = [row for row in rows if row['label'] == label]
        class_kept = sum(row['accepted'] for row in group)
        class_correct = sum(row['acceptedCorrect'] for row in group)
        per_class[label] = {'total': len(group), 'acceptedCorrect': class_correct,
                            'acceptedWrong': class_kept - class_correct,
                            'rejected': len(group) - class_kept}
    report['perClass'] = per_class
    report['confusion'] = {'rows': CLASS_NAMES, 'columns': CLASS_NAMES + ['uncertain'], 'counts': confusion}
    return report


def write_csv_once(path, rows):
    if not rows:
        raise ValueError('Cannot write an empty report.')
    with Path(path).open('x', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def read_frozen_selection(path, expected_sha256):
    if sha256_file(path) != expected_sha256:
        raise RuntimeError('Saved selection changed. Start a new run; do not retune during evaluation.')
    return json.loads(Path(path).read_text(encoding='utf-8'))

## Step 4: choose settings using `val/` and OOD tune only

The grid is 0.00 to 1.00, in steps of 0.01, for each threshold. Among candidates meeting the validation retention target, maximize OOD tuning rejection. Ties prefer more mango retention, then the lower margin, then the higher entropy limit. This deterministic tie-break does not claim an unmeasured generalization benefit.

A margin of zero is valid. With normalized probabilities, confidence >= 0.70 already implies margin >= 0.40 (apart from floating-point rounding). Forcing a nonzero margin does not create an independent detector. The diagnostic below measures its incremental effect.

No test/report predictions enter this stage, including any precision or boundary adjustment. Current production thresholds are a fixed comparison baseline, never selected using evaluation results.

In [9]:
SELECTION_PATH = RUN_DIR / 'selected_thresholds.json'
if SELECTION_PATH.exists():
    raise RuntimeError('Selection already saved. Rerun from Step 1 for a new experiment.')
val_stats = probability_stats(predict_records(val_records))
ood_tune_stats = probability_stats(predict_records(tune_records))
chosen, grid_results = select_thresholds(val_stats, ood_tune_stats, GRID,
                                        CONFIDENCE_THRESHOLD, MIN_ID_RETENTION)
val_rows = scored_rows(val_records, val_stats, chosen)
tune_rows = scored_rows(tune_records, ood_tune_stats, chosen)
val_metrics, tune_metrics = summarize(val_rows), summarize(tune_rows)
without_margin = {**chosen, 'marginThreshold': 0.0}
margin_added = {
    'val': sum(accepted(s, without_margin) and not accepted(s, chosen) for s in val_stats),
    'ood_tune': sum(accepted(s, without_margin) and not accepted(s, chosen) for s in ood_tune_stats)}
selection = {
    'notebookRevision': 'v3.1', 'sourceModelVersion': 'model-v3.0.0',
    'candidateModelVersion': CANDIDATE_VERSION, 'trial': '3', 'classes': CLASS_NAMES,
    'thresholds': chosen, 'baselineThresholds': BASELINE_THRESHOLDS,
    'temperature': TEMPERATURE, 'temperatureFoldedIntoWeights': True,
    'modelFileSha256': MODEL_SHA256, 'resumeInfoSha256': INFO_SHA256,
    'loadedWeightsSha256': WEIGHTS_SHA256, 'inputManifestSha256': MANIFEST_SHA256,
    'runtime': RUNTIME, 'layerPolicies': LAYER_POLICIES,
    'preprocessing': 'Pillow RGB; TensorFlow bilinear 224x224, no antialias; raw float32 0-255',
    'minimumValidationRetention': MIN_ID_RETENTION, 'grid': GRID,
    'objective': 'max OOD tune rejection subject to validation retention floor',
    'tieBreak': 'max validation retention, min margin, max entropy',
    'evaluationStatus': EVALUATION_STATUS, 'evaluationNotes': EVALUATION_NOTES,
    'validationMetrics': val_metrics, 'oodTuneMetrics': tune_metrics,
    'additionalRejectionsFromMargin': margin_added,
    'selectedAtUtc': datetime.now(timezone.utc).isoformat()}
if weight_digest() != WEIGHTS_SHA256:
    raise RuntimeError('Loaded model weights changed during tuning inference.')
write_csv_once(RUN_DIR / 'threshold_search.csv', grid_results)
write_csv_once(RUN_DIR / 'tuning_predictions.csv', val_rows + tune_rows)
write_json_once(SELECTION_PATH, selection)
SELECTION_SHA256 = sha256_file(SELECTION_PATH)
print('SELECTED AND SAVED:', chosen)
print(f"Validation accepted: {val_metrics['accepted']}/{val_metrics['total']} ({val_metrics['retention']:.2%})")
print(f"OOD tune rejected: {tune_metrics['rejected']}/{tune_metrics['total']} ({tune_metrics['rejectionRate']:.2%})")
print('Additional margin rejections:', margin_added)
print('Selection SHA-256:', SELECTION_SHA256)

SELECTED AND SAVED: {'confidenceThreshold': 0.7, 'marginThreshold': 0.0, 'entropyThreshold': 0.44}
Validation accepted: 372/391 (95.14%)
OOD tune rejected: 67/84 (79.76%)
Additional margin rejections: {'val': 0, 'ood_tune': 0}
Selection SHA-256: 12d3b9b42c66b8bf13820d3be0314f213d156afbe0469bc9efded7402332c82e


## Step 5: evaluate the saved settings without changing them

This is the first stage that runs predictions on mango `test/` and OOD report images. It reads the saved selection and manifest, checks identity, and scores all evaluation images. There is no threshold search in this cell.

The predeclared v3 baseline is reported alongside the candidate for context. Do not choose between them or adjust thresholds from these evaluation scores and then claim this same collection is an independent final test.

A genuine mango image rejected as uncertain is a refusal to diagnose. Report it separately from an accepted wrong diagnosis. Small counts and previously reused data limit what the percentages prove.

In [10]:
if (RUN_DIR / 'evaluation.json').exists():
    raise RuntimeError('Evaluation already recorded. Start a new run instead of overwriting it.')
frozen = read_frozen_selection(SELECTION_PATH, SELECTION_SHA256)
if sha256_file(MANIFEST_PATH) != frozen['inputManifestSha256']:
    raise RuntimeError('Image manifest changed after selection.')
if sha256_file(MODEL_PATH) != frozen['modelFileSha256'] or weight_digest() != frozen['loadedWeightsSha256']:
    raise RuntimeError('Model changed after selection.')
if sha256_file(INFO_PATH) != frozen['resumeInfoSha256'] or CLASS_NAMES != frozen['classes']:
    raise RuntimeError('Model metadata/class order changed after selection.')
manifest_for_evaluation = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
verify_records_unchanged(manifest_for_evaluation['images'])
eval_test_records = [r for r in manifest_for_evaluation['images'] if r['subset'] == 'test']
eval_ood_records = [r for r in manifest_for_evaluation['images'] if r['subset'] == 'ood_report']
test_stats = probability_stats(predict_records(eval_test_records))
report_stats = probability_stats(predict_records(eval_ood_records))
evaluation = {'selectionSha256': SELECTION_SHA256,
              'evaluationStatus': frozen['evaluationStatus'], 'evaluationNotes': frozen['evaluationNotes'],
              'measurementRuntime': 'Keras inference; TensorFlow.js browser parity not measured',
              'evaluatedAtUtc': datetime.now(timezone.utc).isoformat()}
for name, thresholds in [('candidate', frozen['thresholds']), ('baseline', frozen['baselineThresholds'])]:
    mango_rows = scored_rows(eval_test_records, test_stats, thresholds)
    ood_rows = scored_rows(eval_ood_records, report_stats, thresholds)
    mango_summary, ood_summary = summarize(mango_rows), summarize(ood_rows)
    evaluation[name] = {'thresholds': thresholds, 'mangoTest': mango_summary, 'oodReport': ood_summary}
    write_csv_once(RUN_DIR / f'{name}_evaluation_predictions.csv', mango_rows + ood_rows)
    print('\n' + name.upper())
    print('Thresholds:', thresholds)
    print(f"Mango accepted: {mango_summary['accepted']}/{mango_summary['total']}; "
          f"correct: {mango_summary['acceptedCorrect']}; wrong: {mango_summary['acceptedWrong']}; "
          f"rejected: {mango_summary['rejected']}")
    accuracy = mango_summary['acceptedAccuracy']
    print('Accuracy among accepted:', f'{accuracy:.2%}' if accuracy is not None else 'N/A')
    print(f"OOD rejected: {ood_summary['rejected']}/{ood_summary['total']} "
          f"({ood_summary['rejectionRate']:.2%}); incorrectly accepted: {ood_summary['accepted']}")
    print('Per disease: total / accepted correct / accepted wrong / rejected')
    for label, counts in mango_summary['perClass'].items():
        print(label, '/', counts['total'], '/', counts['acceptedCorrect'], '/',
              counts['acceptedWrong'], '/', counts['rejected'])
if weight_digest() != frozen['loadedWeightsSha256']:
    raise RuntimeError('Loaded model weights changed during evaluation.')
read_frozen_selection(SELECTION_PATH, SELECTION_SHA256)
write_json_once(RUN_DIR / 'evaluation.json', evaluation)
print('\nEvaluation status:', evaluation['evaluationStatus'])
print(evaluation['evaluationNotes'])


CANDIDATE
Thresholds: {'confidenceThreshold': 0.7, 'marginThreshold': 0.0, 'entropyThreshold': 0.44}
Mango accepted: 374/393; correct: 371; wrong: 3; rejected: 19
Accuracy among accepted: 99.20%
OOD rejected: 51/84 (60.71%); incorrectly accepted: 33
Per disease: total / accepted correct / accepted wrong / rejected
anthracnose / 73 / 73 / 0 / 0
bacterial_black_spot / 45 / 34 / 2 / 9
bacterial_canker / 74 / 73 / 0 / 1
healthy / 92 / 90 / 1 / 1
mango_scab / 34 / 27 / 0 / 7
sooty_mold / 75 / 74 / 0 / 1

BASELINE
Thresholds: {'confidenceThreshold': 0.7, 'marginThreshold': 0.4, 'entropyThreshold': 0.4}
Mango accepted: 373/393; correct: 370; wrong: 3; rejected: 20
Accuracy among accepted: 99.20%
OOD rejected: 57/84 (67.86%); incorrectly accepted: 27
Per disease: total / accepted correct / accepted wrong / rejected
anthracnose / 73 / 73 / 0 / 0
bacterial_black_spot / 45 / 34 / 2 / 9
bacterial_canker / 74 / 73 / 0 / 1
healthy / 92 / 89 / 1 / 2
mango_scab / 34 / 27 / 0 / 7
sooty_mold / 75 / 74 

## Step 6: export the candidate and download the evidence

The JSON below is an app-compatible **candidate** configuration, not a deployment. It records newly measured results rather than carrying forward historical accuracy or simulated browser-precision claims. A selected threshold may equal the current value; a change is not guaranteed to improve accuracy.

Review OOD labels, evaluation errors, the retention/rejection trade-off, and browser behavior before adopting it. Actual deployment is a separate repository change and build.

The ZIP contains the candidate config, frozen selection, evaluation, search table, per-image predictions, input manifest, and a short report. It contains no image copies or model weights. Paths in the manifest identify your own Drive files, so review them before sharing.

In [11]:
from google.colab import files

frozen = read_frozen_selection(SELECTION_PATH, SELECTION_SHA256)
measured = json.loads((RUN_DIR / 'evaluation.json').read_text(encoding='utf-8'))
if measured['selectionSha256'] != SELECTION_SHA256:
    raise RuntimeError('Evaluation belongs to a different threshold selection.')
final_mango = measured['candidate']['mangoTest']
final_ood = measured['candidate']['oodReport']
config = {
    'modelVersion': frozen['candidateModelVersion'], 'sourceModelVersion': frozen['sourceModelVersion'],
    'notebookRevision': frozen['notebookRevision'], 'trial': frozen['trial'], 'classes': frozen['classes'],
    'inputSize': [224, 224], 'pixelRange': '0-255 raw, MobileNetV3 rescales internally',
    'temperature': frozen['temperature'], 'temperatureFoldedIntoWeights': True,
    'thresholds': frozen['thresholds'],
    'rejection': {
        'validationRetention': frozen['validationMetrics']['retention'],
        'tuneRejection': frozen['oodTuneMetrics']['rejectionRate'],
        'testRetention': final_mango['retention'], 'accuracyOnAccepted': final_mango['acceptedAccuracy'],
        'reportRejection': final_ood['rejectionRate'],
        'validationCount': frozen['validationMetrics']['total'], 'testCount': final_mango['total'],
        'oodTuneCount': frozen['oodTuneMetrics']['total'], 'oodReportCount': final_ood['total'],
        'evaluationStatus': frozen['evaluationStatus'], 'evaluationNotes': frozen['evaluationNotes']},
    'provenance': {'sourceKerasSha256': frozen['modelFileSha256'],
                   'inputManifestSha256': frozen['inputManifestSha256'],
                   'selectionSha256': SELECTION_SHA256,
                   'browserParityMeasured': False, 'deploymentStatus': 'candidate'}}
write_json_once(RUN_DIR / 'model_config.json', config)
accuracy = final_mango['acceptedAccuracy']
accuracy_text = f'{accuracy:.2%}' if accuracy is not None else 'N/A (none accepted)'
report_lines = [
    '# Threshold-fitting notebook v3.1: run report', '',
    f"- Source weights: {frozen['sourceModelVersion']}; inference only, no retraining.",
    f"- Candidate config label: {frozen['candidateModelVersion']}.",
    f"- Selected thresholds: {json.dumps(frozen['thresholds'])}.",
    '- Selection used validation mango images and OOD tune only.',
    f"- Validation retention: {frozen['validationMetrics']['retention']:.2%}.",
    f"- Mango evaluation: {final_mango['total']} images; {final_mango['acceptedCorrect']} accepted correct, "
    f"{final_mango['acceptedWrong']} accepted wrong, {final_mango['rejected']} rejected.",
    f'- Accuracy among accepted mango predictions: {accuracy_text}.',
    f"- OOD evaluation: {final_ood['rejected']}/{final_ood['total']} rejected "
    f"({final_ood['rejectionRate']:.2%}); {final_ood['accepted']} incorrectly accepted.",
    f"- Evaluation provenance: {frozen['evaluationStatus']}. {frozen['evaluationNotes']}",
    '- Browser parity has not been measured. These are Keras runtime results.', '',
    'The retention target applies to validation selection, not a guarantee about evaluation or farm photos.',
    'No evaluation result changed the selected thresholds. Review labels, per-image errors, and physical-phone',
    'behavior before deploying the candidate. Keep completed v3 notebooks and save this run under a new name.', '']
report_text = '\n'.join(report_lines)
with (RUN_DIR / 'REPORT.md').open('x', encoding='utf-8') as handle:
    handle.write(report_text)
archive_path = RUN_DIR / 'threshold_v3_1_results.zip'
with zipfile.ZipFile(archive_path, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.iterdir()):
        if path.is_file() and path != archive_path:
            archive.write(path, arcname=path.name)
print(report_text)
print('Files saved in:', RUN_DIR)
files.download(str(archive_path))

# Threshold-fitting notebook v3.1: run report

- Source weights: model-v3.0.0; inference only, no retraining.
- Candidate config label: model-v3.0.1.
- Selected thresholds: {"confidenceThreshold": 0.7, "marginThreshold": 0.0, "entropyThreshold": 0.44}.
- Selection used validation mango images and OOD tune only.
- Validation retention: 95.14%.
- Mango evaluation: 393 images; 371 accepted correct, 3 accepted wrong, 19 rejected.
- Accuracy among accepted mango predictions: 99.20%.
- OOD evaluation: 51/84 rejected (60.71%); 33 incorrectly accepted.
- Evaluation provenance: previously_used. Original v3 test/report images influenced earlier threshold work.
- Browser parity has not been measured. These are Keras runtime results.

The retention target applies to validation selection, not a guarantee about evaluation or farm photos.
No evaluation result changed the selected thresholds. Review labels, per-image errors, and physical-phone
behavior before deploying the candidate. Keep completed v3

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Method references

- [Keeping model selection separate from final testing](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Keras model loading](https://www.tensorflow.org/api_docs/python/tf/keras/models/load_model)
- [TensorFlow image resizing](https://www.tensorflow.org/api_docs/python/tf/image/resize)
- [MobileNetV3 input preprocessing](https://keras.io/api/applications/mobilenet/)

The original v3 notebooks are historical evidence. This revision changes the threshold-fitting procedure, not the original recorded results.